# Домашнє завдання 8 · Stable Diffusion: генерація, гіперпараметри та адаптація DreamBooth-LoRA

**Заняття 7–8 · генеративні моделі, адаптація та керування · ≈ 3 години · 15 балів**

Розбивка балів — у README завдання. Бонусна частина 4 балів не додає.

У цьому завданні ви пройдете шлях від запуску готового пайплайна Stable Diffusion 1.5 до власної адаптації моделі під конкретний об'єкт або стиль:

| Частина | Що робимо | Орієнтовний час |
|---|---|---|
| 0 | Налаштування середовища | 5 хв |
| 1 | Базова генерація, функція `generate` | 10 хв |
| 2 | Експерименти: seed, guidance scale, кроки, семплери, negative prompt, CFG власноруч | 40–60 хв |
| 3 | DreamBooth-LoRA на власних зображеннях, порівняння до/після, сила LoRA, чекпоінти | 60–90 хв |
| 4 | Бонус (без балів): img2img, кількісна оцінка CLIP | 20–30 хв |
| 5 | Висновки та ідеї для самостійної роботи | 15 хв |

**Як працювати з ноутбуком**

* Комірки з позначкою **`TODO N`** містять функції або фрагменти, які потрібно дописати. Не змінюйте сигнатури функцій — наступні комірки на них спираються.
* Комірки з позначкою **✍️** — поля для ваших спостережень і висновків. Відповідайте змістовно, з посиланням на конкретні зображення та числа.
* Уся робота виконується на **Google Colab з GPU T4**. Локальний GPU не потрібен.

**Що здати:** **один архів** `dz8_<прізвище>.zip` в особистому кабінеті LMS. У ньому — цей ноутбук (`.ipynb`) зі збереженими виводами всіх комірок (зображення, таблиці, графіки) та заповненими полями ✍️, і `hw_outputs.zip`, який створює остання комірка частини 5. Перед збереженням перевірте, що зображення відображаються (*File → Download → Download .ipynb*). Окремі файли чи посилання на Drive не приймаються.

## 0. Налаштування середовища

1. *Runtime → Change runtime type → **T4 GPU***.
2. Запустіть комірки нижче по черзі. Якщо після встановлення бібліотек імпорт падає з помилкою версій — виконайте *Runtime → Restart session* і запустіть комірки знову (встановлення вдруге пройде миттєво).

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU не знайдено: Runtime → Change runtime type → T4 GPU"
print(f"GPU:  {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"torch {torch.__version__}")

In [ ]:
# diffusers — пайплайни дифузійних моделей; peft — бекенд LoRA; accelerate — запуск навчання
!pip install -q -U diffusers transformers accelerate peft
# Colab має передвстановлений застарілий torchao, через який новий peft падає при створенні LoRA
# ("Found an incompatible version of torchao"). Для цього завдання torchao не потрібен.
!pip uninstall -y -q torchao

import importlib.metadata as im
for pkg in ["diffusers", "transformers", "accelerate", "peft"]:
    print(f"{pkg:13s} {im.version(pkg)}")

In [ ]:
import sys, gc, time, math, shutil, subprocess, codecs, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

from diffusers import (
    StableDiffusionPipeline, StableDiffusionImg2ImgPipeline,
    PNDMScheduler, DDIMScheduler, EulerDiscreteScheduler,
    EulerAncestralDiscreteScheduler, DPMSolverMultistepScheduler, UniPCMultistepScheduler,
)

# Оригінальний репозиторій runwayml/stable-diffusion-v1-5 видалено з HuggingFace у 2024 р.;
# це офіційне дзеркало з тими самими вагами.
MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"
DEVICE = "cuda"
DTYPE = torch.float16          # float16 вдвічі зменшує використання VRAM

WORK_DIR = Path("/content/hw_genmodels")
OUT_DIR = WORK_DIR / "outputs"          # сюди зберігаються всі фігури для звіту
OUT_DIR.mkdir(parents=True, exist_ok=True)

TIMINGS = []   # журнал вимірювань часу генерації (заповнюється у run_sweep)

### Допоміжні функції (готові, змінювати не потрібно)

In [ ]:
def show_images(images, titles=None, suptitle=None, cols=None, size=4, save_as=None):
    """Показує список PIL-зображень сіткою і (опційно) зберігає фігуру в OUT_DIR."""
    n = len(images)
    cols = cols or n
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * rows), squeeze=False)
    for i, ax in enumerate(axes.flat):
        ax.axis("off")
        if i < n:
            ax.imshow(images[i])
            if titles is not None:
                ax.set_title(titles[i], fontsize=9)
    if suptitle:
        fig.suptitle(suptitle, fontsize=12)
    fig.tight_layout()
    if save_as:
        fig.savefig(OUT_DIR / save_as, dpi=100, bbox_inches="tight")
    plt.show()


def log_timing(experiment, param, value, seconds, **extra):
    """Додає запис у глобальний журнал TIMINGS."""
    TIMINGS.append({"experiment": experiment, "param": param, "value": str(value),
                    "seconds": round(seconds, 3), **extra})


def free_memory():
    gc.collect()
    torch.cuda.empty_cache()
    print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


def load_pipeline(model_id=MODEL_ID):
    """Завантажує SD 1.5 у float16 на GPU (~4 GB при першому запуску, далі — з кешу)."""
    pipe = StableDiffusionPipeline.from_pretrained(
        model_id, torch_dtype=DTYPE,
        safety_checker=None, requires_safety_checker=False,  # вимкнено для навчальних цілей
    ).to(DEVICE)
    pipe.set_progress_bar_config(disable=True)
    return pipe

## 1. Базова генерація

Нагадування з лекції. Stable Diffusion — латентна дифузійна модель із трьох компонентів:

* **VAE** стискає зображення 512×512×3 у латент 64×64×4 і декодує назад;
* **CLIP text encoder** перетворює промпт у послідовність ембедінгів;
* **UNet** передбачає шум $\varepsilon_\theta(z_t, t, c)$ у латентному просторі й отримує текст через cross-attention.

Генерація: $z_T \sim \mathcal N(0, I)$ → семплер $N$ разів викликає UNet і крок за кроком прибирає шум → VAE декодує $z_0$ у зображення.

In [ ]:
pipe = load_pipeline()
print("Scheduler за замовчуванням:", type(pipe.scheduler).__name__)

### TODO 1 — функція `generate`

Напишіть обгортку над пайплайном, яка:
1. фіксує випадковість через генератор PyTorch, створений на GPU та ініціалізований заданим `seed`, — це робить результат відтворюваним;
2. вимірює **час** виклику пайплайна. Врахуйте, що операції на CUDA виконуються асинхронно: Python-виклик може повернутися раніше, ніж GPU завершить обчислення. Знайдіть у `torch.cuda` спосіб дочекатися завершення і використайте його біля обох точок вимірювання;
3. повертає пару `(PIL.Image, секунди)`.

In [ ]:
def generate(pipe, prompt, seed=42, steps=30, guidance=7.5,
             negative_prompt=None, height=512, width=512, **kwargs):
    """Генерує одне зображення. Повертає (image: PIL.Image, elapsed: float, секунди).

    TODO 1:
      * створіть генератор випадкових чисел PyTorch на пристрої DEVICE та ініціалізуйте його seed;
      * засічіть час до і після виклику пайплайна (дочекайтесь завершення CUDA-обчислень);
      * передайте пайплайну промпт, negative prompt, кількість кроків, guidance scale, розмір
        зображення, генератор і додаткові kwargs (назви аргументів — у документації
        StableDiffusionPipeline.__call__);
      * пайплайн повертає об'єкт зі списком зображень — вам потрібне перше.
    """
    raise NotImplementedError("TODO 1: реалізуйте generate")

In [ ]:
# Перевірка TODO 1: однаковий seed → (майже) однакові зображення
_ = generate(pipe, "warm-up", steps=2)   # перший виклик включає ініціалізацію CUDA-ядер

img_a, t_a = generate(pipe, "a red apple on a wooden table", seed=0, steps=20)
img_b, _   = generate(pipe, "a red apple on a wooden table", seed=0, steps=20)
img_c, _   = generate(pipe, "a red apple on a wooden table", seed=1, steps=20)

diff_same = np.abs(np.asarray(img_a, np.int16) - np.asarray(img_b, np.int16)).mean()
diff_other = np.abs(np.asarray(img_a, np.int16) - np.asarray(img_c, np.int16)).mean()
print(f"Час генерації (20 кроків): {t_a:.2f} с")
print(f"Середня різниця пікселів: той самий seed = {diff_same:.2f}, інший seed = {diff_other:.2f}")
assert diff_same < 2.0 < diff_other, "Перевірте, чи правильно передається generator"
show_images([img_a, img_b, img_c], ["seed=0", "seed=0 (повтор)", "seed=1"], cols=3, size=3)

### Базова генерація: щонайменше 3 промпти

Замініть або доповніть список власними промптами. Спробуйте різні типи сцен: фотореалізм, ілюстрацію, архітектуру, людей, текст на зображенні.

In [ ]:
BASE_PROMPTS = [
    "a photograph of an astronaut riding a horse on Mars, photorealistic, 4k",
    "an isometric illustration of a tiny cozy library, soft pastel colors",
    "a street in Kyiv after rain at night, neon reflections, cinematic photo",
    # додайте власні промпти
]

base_images = []
for i, p in enumerate(BASE_PROMPTS):
    img, t = generate(pipe, p, seed=i)
    base_images.append(img)
    print(f"[{t:.2f} s] {p}")

show_images(base_images, [p[:45] + "…" for p in BASE_PROMPTS], cols=3,
            suptitle="Базова генерація SD 1.5", save_as="part1_base.png")

## 2. Експерименти з параметрами генерації

У кожному експерименті змінюється **лише один** параметр, решта (зокрема seed) фіксовані. Це дозволяє ізолювати його вплив.

### TODO 2 — функція `run_sweep`

Функція проходить по значеннях `values` параметра `param`, для кожного викликає `generate` і записує час у журнал через `log_timing`.

In [ ]:
def run_sweep(pipe, base_kwargs, param, values, experiment=None):
    """Перебирає значення одного параметра generate().

    base_kwargs — словник аргументів generate (prompt, seed, steps, ...);
    param — назва аргументу, що змінюється (наприклад "guidance");
    values — список значень.
    Повертає (images, times).

    TODO 2: для кожного значення зі списку values
      * сформуйте аргументи generate: базові аргументи + поточне значення параметра param
        (не змінюйте сам словник base_kwargs — він спільний для всіх ітерацій);
      * згенеруйте зображення і збережіть зображення та час у відповідні списки;
      * запишіть вимір через log_timing. Назва експерименту — experiment, а якщо його не задано,
        то назва параметра. Як додаткові поля передайте steps (фактична кількість кроків,
        за замовчуванням у generate це 30) і scheduler (назва класу поточного семплера).
    """
    images, times = [], []
    raise NotImplementedError("TODO 2: реалізуйте run_sweep")
    return images, times

### Експеримент А — вплив seed

Seed визначає початковий шум $z_T$. За однакового промпту та параметрів різні seed дають різні композиції.

In [ ]:
seeds = [0, 1, 2, 42, 1337]
imgs, times = run_sweep(pipe, dict(prompt="a lighthouse on a cliff during a storm, dramatic oil painting"),
                        "seed", seeds, experiment="A_seed")
show_images(imgs, [f"seed={s}\n{t:.2f}s" for s, t in zip(seeds, times)],
            suptitle="Експеримент А: різні seed", save_as="expA_seed.png")

### Експеримент Б — вплив `guidance_scale` (CFG)

На кожному кроці UNet робить два передбачення — умовне та безумовне — і вони комбінуються:

$$\hat\varepsilon = \varepsilon_{\text{uncond}} + w\,(\varepsilon_{\text{cond}} - \varepsilon_{\text{uncond}})$$

Зверніть увагу: при $w = 1$ отримуємо просто умовне передбачення (без підсилення), а при $w = 0$ — безумовне, що ігнорує промпт. У `diffusers` при `guidance_scale ≤ 1` CFG вимикається зовсім і UNet викликається один раз замість двох — перевірте це за часом генерації.

In [ ]:
guidance_values = [1.0, 3.0, 7.5, 12.0, 20.0]
imgs, times = run_sweep(pipe, dict(prompt="a portrait of a cat as a medieval knight, oil painting, detailed",
                                   seed=1337),
                        "guidance", guidance_values, experiment="B_guidance")
show_images(imgs, [f"cfg={g}\n{t:.2f}s" for g, t in zip(guidance_values, times)],
            suptitle="Експеримент Б: guidance_scale (seed фіксований)", save_as="expB_guidance.png")

### Експеримент В — кількість кроків семплера

In [ ]:
steps_list = [5, 10, 20, 30, 50]
imgs, times = run_sweep(pipe, dict(prompt="a serene Japanese garden with cherry blossoms, watercolor", seed=42),
                        "steps", steps_list, experiment="C_steps")
show_images(imgs, [f"steps={s}\n{t:.2f}s" for s, t in zip(steps_list, times)],
            suptitle="Експеримент В: num_inference_steps", save_as="expC_steps.png")

plt.figure(figsize=(5, 3))
plt.plot(steps_list, times, "o-")
plt.xlabel("кроки"); plt.ylabel("час, с"); plt.title("Час генерації vs кількість кроків"); plt.grid(alpha=.3)
plt.savefig(OUT_DIR / "expC_time.png", dpi=100, bbox_inches="tight"); plt.show()
print("Час на крок, с:", [round(t / s, 3) for s, t in zip(steps_list, times)])

### TODO 3 — семплери (schedulers)

Семплер визначає, як із передбаченого шуму обчислюється наступний латент і на яких кроках $t$ викликається UNet. Одна й та сама навчена модель може працювати з різними семплерами без перенавчання.

Реалізуйте `set_scheduler`: за назвою знайдіть у словнику `SCHEDULERS` клас семплера та його додаткові параметри, створіть новий семплер **на основі конфігурації оригінального** (вона збережена в `DEFAULT_SCHEDULER_CONFIG`; у класах семплерів diffusers є метод, що створює об'єкт з конфігурації й дозволяє перевизначити окремі поля) і замініть ним семплер пайплайна.

Чому саме з конфігурації оригінального семплера: у ній зберігається розклад шуму (betas), з яким навчалась модель. Семплер із типовими налаштуваннями матиме інший розклад і генеруватиме некоректні зображення.

In [ ]:
SCHEDULERS = {
    "PNDM (default)":  (PNDMScheduler, {}),
    "DDIM":            (DDIMScheduler, {}),
    "Euler":           (EulerDiscreteScheduler, {}),
    "Euler a":         (EulerAncestralDiscreteScheduler, {}),
    "DPM++ 2M":        (DPMSolverMultistepScheduler, {}),
    "DPM++ 2M Karras": (DPMSolverMultistepScheduler, {"use_karras_sigmas": True}),
    "UniPC":           (UniPCMultistepScheduler, {}),
}
DEFAULT_SCHEDULER_CONFIG = pipe.scheduler.config   # зберігаємо оригінальну конфігурацію


def set_scheduler(pipe, name):
    """TODO 3: встановіть семплер пайплайна за назвою з SCHEDULERS (див. опис вище)."""
    raise NotImplementedError("TODO 3: реалізуйте set_scheduler")
    return pipe


set_scheduler(pipe, "DDIM")
assert isinstance(pipe.scheduler, DDIMScheduler)
set_scheduler(pipe, "PNDM (default)")
print("TODO 3 OK")

### Експеримент Г — семплери × кількість кроків

Кожен рядок — семплер, кожен стовпчик — кількість кроків. Зверніть увагу, за скільки кроків кожен семплер дає прийнятну якість і чи «стабілізується» зображення зі збільшенням кроків. Займає 2–3 хвилини.

In [ ]:
sampler_prompt = dict(prompt="a red fox in a snowy forest, golden hour, highly detailed photo", seed=7)
sampler_steps = [10, 20, 30]

grid_imgs, grid_titles = [], []
for name in SCHEDULERS:
    set_scheduler(pipe, name)
    imgs, times = run_sweep(pipe, sampler_prompt, "steps", sampler_steps, experiment=f"D_sampler:{name}")
    grid_imgs += imgs
    grid_titles += [f"{name} | {s} steps | {t:.2f}s" for s, t in zip(sampler_steps, times)]

set_scheduler(pipe, "PNDM (default)")   # повертаємо семплер за замовчуванням
show_images(grid_imgs, grid_titles, cols=len(sampler_steps), size=3.2,
            suptitle="Експеримент Г: семплери × кроки", save_as="expD_samplers.png")

In [ ]:
df_sampler = pd.DataFrame([r for r in TIMINGS if r["experiment"].startswith("D_sampler")])
df_sampler["sampler"] = df_sampler["experiment"].str.replace("D_sampler:", "", regex=False)
print("Час генерації (с):")
display(df_sampler.pivot_table(index="sampler", columns="value", values="seconds").round(2))

### Експеримент Ґ — negative prompt

У `diffusers` негативний промпт підставляється замість порожнього промпту в безумовну гілку CFG, тобто $\varepsilon_{\text{uncond}}$ стає $\varepsilon_{\text{neg}}$ і генерація відштовхується від описаних у ньому ознак.

In [ ]:
negative = "blurry, low quality, dark, distorted, watermark, text, ugly"
imgs, times = run_sweep(pipe, dict(prompt="a cozy wooden cabin in the forest, sunset, warm lighting", seed=7),
                        "negative_prompt", [None, negative], experiment="E_negative")
show_images(imgs, ["без negative prompt", "з negative prompt"], cols=2, size=5,
            suptitle="Експеримент Ґ: negative prompt", save_as="expE_negative.png")

### TODO 4 — CFG власноруч

Щоб побачити механіку дифузії зсередини, напишіть цикл розшумлення самостійно, використовуючи компоненти пайплайна (`pipe.unet`, `pipe.scheduler`, `pipe.vae`). Кодування тексту, ініціалізація латента й декодування вже написані — вам потрібно реалізувати **тіло циклу** для кроку $t$:

1. Обидві гілки CFG (безумовну й умовну) обчисліть **одним** прогоном UNet: сформуйте батч із двох копій поточного латента.
2. Деякі семплери вимагають масштабувати вхід моделі на кожному кроці — у семплера для цього є окремий метод.
3. Прогоніть UNet на батчі. Крім латента й кроку, мережа приймає текстові ембедінги (аргумент `encoder_hidden_states`) — теж батчем із двох, **у тому ж порядку**, що й латенти. UNet повертає об'єкт, з якого треба взяти передбачений шум.
4. Розділіть передбачення на безумовну та умовну частини й скомбінуйте їх за формулою CFG з розділу про експеримент Б.
5. Додайте у `delta_norms` норму різниці умовного та безумовного передбачень (як число Python).
6. Зробіть крок семплера: він отримує скомбінований шум, крок $t$ і поточний латент і повертає об'єкт із латентом для наступного кроку.

Корисно переглянути, як це зроблено в `StableDiffusionPipeline.__call__` у вихідному коді diffusers. Якщо все правильно, результат збігатиметься з `pipe(...)` при тому ж seed.

In [ ]:
@torch.no_grad()
def manual_cfg_sample(pipe, prompt, seed=42, steps=30, guidance=7.5,
                      negative_prompt=None, height=512, width=512):
    """Власний цикл семплювання з classifier-free guidance.
    Повертає (PIL.Image, delta_norms: list[float])."""
    # 1. Текстові ембедінги: умовні (промпт) та безумовні ("" або negative prompt)
    cond_emb, uncond_emb = pipe.encode_prompt(prompt, DEVICE, 1, True, negative_prompt)

    # 2. Сітка кроків семплера
    pipe.scheduler.set_timesteps(steps, device=DEVICE)

    # 3. Початковий шум z_T (так само, як усередині пайплайна)
    generator = torch.Generator(DEVICE).manual_seed(seed)
    shape = (1, pipe.unet.config.in_channels,
             height // pipe.vae_scale_factor, width // pipe.vae_scale_factor)
    latents = torch.randn(shape, generator=generator, device=DEVICE, dtype=cond_emb.dtype)
    latents = latents * pipe.scheduler.init_noise_sigma

    delta_norms = []
    # 4. Цикл розшумлення
    for t in pipe.scheduler.timesteps:
        # TODO 4: кроки 1–6 з опису вище
        raise NotImplementedError("TODO 4: реалізуйте крок CFG")

    # 5. Декодування латента VAE
    image = pipe.vae.decode(latents / pipe.vae.config.scaling_factor).sample
    return pipe.image_processor.postprocess(image, output_type="pil")[0], delta_norms

In [ ]:
cfg_prompt = "a portrait of a cat as a medieval knight, oil painting, detailed"
img_pipe, _ = generate(pipe, cfg_prompt, seed=1337, steps=30, guidance=7.5)
img_manual, norms = manual_cfg_sample(pipe, cfg_prompt, seed=1337, steps=30, guidance=7.5)

diff = np.abs(np.asarray(img_pipe, np.int16) - np.asarray(img_manual, np.int16)).mean()
print(f"Середня різниця pipe vs manual: {diff:.3f} (очікується < 1)")
show_images([img_pipe, img_manual], ["pipe(...)", "manual_cfg_sample"], cols=2, size=4)

# w = 0 — лише безумовна гілка (промпт ігнорується); w = 1 — лише умовна, без підсилення
imgs_w = [manual_cfg_sample(pipe, cfg_prompt, seed=1337, guidance=w)[0] for w in [0.0, 1.0, 7.5]]
show_images(imgs_w, ["w=0 (uncond)", "w=1 (cond, без підсилення)", "w=7.5"], cols=3, size=4,
            suptitle="CFG власноруч: крайні значення w", save_as="expF_manual_cfg.png")

plt.figure(figsize=(5, 3))
plt.plot(norms, "o-", ms=3)
plt.xlabel("номер кроку (від шуму до зображення)"); plt.ylabel("‖ε_cond − ε_uncond‖")
plt.title("Внесок тексту протягом семплювання"); plt.grid(alpha=.3)
plt.savefig(OUT_DIR / "expF_delta_norms.png", dpi=100, bbox_inches="tight"); plt.show()

In [ ]:
# Зведена таблиця часу всіх експериментів (для звіту)
df_timings = pd.DataFrame(TIMINGS)
df_timings.to_csv(OUT_DIR / "timings.csv", index=False)
display(df_timings.groupby("experiment")["seconds"].agg(["count", "mean", "min", "max"]).round(2))

### ✍️ Аналіз частини 2

Для кожного пункту дайте 2–3 речення: **що** змінилось на зображеннях і **чому** з погляду алгоритму. Посилайтеся на конкретні зображення та числа з таблиць.

1. **Seed.** Що саме змінюється між seed, а що лишається спільним? Що фізично задає seed?
2. **Guidance scale.** Як змінюються відповідність промпту, насиченість кольорів, артефакти, різноманітність? Чому генерація з `cfg=1.0` приблизно вдвічі швидша?
3. **Кроки.** Після якої кількості кроків якість перестає помітно зростати? Як залежить час від кількості кроків?
4. **Семплери.** Які семплери дають прийнятну якість уже на 10–20 кроках? Чому «Euler a» не збігається до одного зображення зі збільшенням кроків? Чи відрізняється час одного кроку між семплерами, і що тоді визначає вартість генерації?
5. **Negative prompt.** Чи покращився результат? Поясніть через формулу CFG.
6. **CFG власноруч.** Чи збігся результат із пайплайном? Чим відрізняються зображення при $w=0$, $w=1$, $w=7.5$? Як змінюється $\lVert\varepsilon_{\text{cond}} - \varepsilon_{\text{uncond}}\rVert$ протягом семплювання і що це говорить про те, *коли* текст найбільше впливає на зображення?

✍️ **Ваші відповіді:**

1. …
2. …
3. …
4. …
5. …
6. …

## 3. Адаптація: DreamBooth-LoRA

**DreamBooth** вчить модель пов'язувати рідкісний ідентифікатор (тут `sks`) з конкретним об'єктом або стилем за 5–15 зображеннями. **LoRA** замість оновлення всіх ваг UNet навчає низькорангову поправку $W = W_0 + BA$, де $B \in \mathbb R^{d\times r}$, $A \in \mathbb R^{r\times k}$, $r \ll \min(d,k)$. Оригінальні ваги заморожені, адаптер займає кілька мегабайт.

### 3.1 Вибір об'єкта та датасет

Оберіть **один** варіант (в ідеалі — на власних даних):

* **Об'єкт** — ваш домашній улюбленець, іграшка, чашка, рюкзак, кросівки, будівля. Потрібно 8–15 фото.
* **Стиль** — певна художня манера (ваші малюнки, піксель-арт, ретро-фото). Потрібно 10–20 зображень.

Якщо власних даних зараз немає, встановіть `DATA_SOURCE = "demo"` — завантажиться демонстраційний набір із 5 фото собаки від HuggingFace. **Максимальну оцінку отримують роботи на власних даних.**

#### Як зняти фото об'єкта

* Об'єкт займає значну частину кадру (після обрізання до квадрата по центру він має лишитися в кадрі повністю — тримайте його по центру).
* Різні ракурси (спереду, збоку, згори, під кутом), різні фони та освітлення. Якщо всі фото на одному столі, модель вивчить стіл разом з об'єктом.
* Без інших людей і предметів, що повторюються з кадру в кадр; без майже однакових кадрів.
* Формат JPEG, PNG, WEBP або HEIC (HEIC з iPhone конвертується автоматично). Бажано ≥ 512 px по меншій стороні — фото з будь-якого сучасного телефона підходять.

#### Як завантажити фото в Colab — три способи

**Спосіб 1 — з комп'ютера через кнопку (`DATA_SOURCE = "upload"`).** Найпростіший.
1. Перенесіть фото з телефона на комп'ютер (кабель, Google Photos, Telegram «Надіслати як файл», щоб не втратити якість).
2. У комірці нижче встановіть `DATA_SOURCE = "upload"` і запустіть її.
3. Під коміркою з'явиться кнопка **Choose Files / Вибрати файли**. Виділіть усі фото одразу (Ctrl+A або Shift/Ctrl+клік) і натисніть «Відкрити».
4. Дочекайтеся, поки для кожного файлу з'явиться `100% done`, і комірка завершиться.

Можна також завантажити один **ZIP-архів** із фото — він розпакується автоматично. Якщо кнопка не з'являється або завантаження зависає (буває в деяких браузерах або при блокуванні сторонніх cookies), скористайтеся способом 2 або 3.

**Спосіб 2 — з Google Drive (`DATA_SOURCE = "drive"`).** Зручно, якщо фото вже в хмарі або ви будете перезапускати сесію.
1. На [drive.google.com](https://drive.google.com) у «Мій диск» створіть папку, наприклад `my_images`, і завантажте туди фото (з телефона — через застосунок Google Drive).
2. Встановіть `DATA_SOURCE = "drive"` і вкажіть шлях у `DRIVE_IMAGES_DIR`: для папки `my_images` у корені «Мій диск» це `/content/drive/MyDrive/my_images`.
3. Запустіть комірку й у спливному вікні дозвольте Colab доступ до Google Drive (вибирайте той самий акаунт, у якому відкрито Colab).

**Спосіб 3 — через файлову панель Colab (`DATA_SOURCE = "folder"`).**
1. Відкрийте панель **Files** (іконка папки на лівій бічній панелі).
2. Натисніть правою кнопкою на `content` → **New folder** → назвіть `my_images`.
3. Перетягніть фото з провідника у цю папку й дочекайтеся завершення завантаження (індикатор унизу панелі).
4. Встановіть `DATA_SOURCE = "folder"` і запустіть комірку.

⚠️ Файли у `/content` видаляються після завершення сесії Colab. Щоб не завантажувати фото щоразу, використовуйте спосіб 2.

Після запуску комірки перевірте кількість знайдених зображень, а після TODO 5 — сітку попереднього перегляду: чи всі фото правильно орієнтовані та чи не обрізано об'єкт.

In [ ]:
DATA_SOURCE = "demo"      # "upload" | "drive" | "folder" | "demo" — див. інструкцію вище
DRIVE_IMAGES_DIR = "/content/drive/MyDrive/my_images"   # для DATA_SOURCE = "drive"
FOLDER_IMAGES_DIR = "/content/my_images"                # для DATA_SOURCE = "folder"

IMG_EXT = {".jpg", ".jpeg", ".png", ".webp", ".heic", ".heif"}
RAW_DIR = WORK_DIR / "raw_images"
shutil.rmtree(RAW_DIR, ignore_errors=True)
RAW_DIR.mkdir(parents=True)


def collect_images(src_dir, dst_dir):
    """Копіює зображення з src_dir (включно з підпапками та ZIP-архівами) у dst_dir."""
    src_dir = Path(src_dir)
    for z in list(src_dir.rglob("*.zip")):
        shutil.unpack_archive(z, z.with_suffix(""))
    n = 0
    for p in sorted(src_dir.rglob("*")):
        hidden = any(part.startswith((".", "__MACOSX")) for part in p.relative_to(src_dir).parts)
        if p.is_file() and not hidden and p.suffix.lower() in IMG_EXT:
            shutil.copy(p, Path(dst_dir) / f"{n:03d}_{p.name}")
            n += 1
    return n


if DATA_SOURCE == "upload":
    from google.colab import files
    upload_dir = WORK_DIR / "uploaded"
    shutil.rmtree(upload_dir, ignore_errors=True)
    upload_dir.mkdir(parents=True)
    for name, data in files.upload().items():
        (upload_dir / name).write_bytes(data)
    collect_images(upload_dir, RAW_DIR)
elif DATA_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    assert Path(DRIVE_IMAGES_DIR).is_dir(), f"Папку {DRIVE_IMAGES_DIR} не знайдено — перевірте шлях"
    collect_images(DRIVE_IMAGES_DIR, RAW_DIR)
elif DATA_SOURCE == "folder":
    assert Path(FOLDER_IMAGES_DIR).is_dir(), f"Папку {FOLDER_IMAGES_DIR} не знайдено — створіть її у панелі Files"
    collect_images(FOLDER_IMAGES_DIR, RAW_DIR)
elif DATA_SOURCE == "demo":
    from huggingface_hub import snapshot_download
    hf_dir = snapshot_download("diffusers/dog-example", repo_type="dataset",
                               local_dir=WORK_DIR / "dog_example_hf")
    collect_images(hf_dir, RAW_DIR)
else:
    raise ValueError(f"Невідомий DATA_SOURCE: {DATA_SOURCE}")

raw_files = sorted(RAW_DIR.iterdir())
if any(p.suffix.lower() in {".heic", ".heif"} for p in raw_files):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pillow-heif"], check=True)
    from pillow_heif import register_heif_opener
    register_heif_opener()      # після цього PIL відкриває HEIC як звичайні зображення

print(f"Знайдено {len(raw_files)} зображень:", [p.name for p in raw_files])
assert len(raw_files) >= 5, "Потрібно щонайменше 5 зображень — перевірте завантаження"

### TODO 5 — підготовка зображень

Навчальний скрипт читає **всі** файли з папки датасету, тому туди мають потрапити лише зображення — без службових файлів. Реалізуйте `center_crop_resize`:

1. врахуйте орієнтацію з EXIF: фото з телефона часто зберігають поворот лише в метаданих, і без цього частина кадрів буде «лежати на боці» (у модулі `PIL.ImageOps` є відповідна функція);
2. приведіть зображення до трьох каналів RGB (PNG можуть мати альфа-канал, а чорно-білі фото — один канал);
3. виріжте з центру квадрат зі стороною, що дорівнює меншій стороні зображення;
4. масштабуйте квадрат до `size × size` якісним фільтром інтерполяції (наприклад, Lanczos).

In [ ]:
def center_crop_resize(img, size=512):
    """TODO 5: EXIF-орієнтація → RGB → центральний квадрат → масштабування до size×size (див. опис вище)."""
    raise NotImplementedError("TODO 5: реалізуйте center_crop_resize")


def prepare_instance_dir(files, dst_dir, size=512):
    """Очищає dst_dir і зберігає туди підготовлені зображення у PNG."""
    dst_dir = Path(dst_dir)
    shutil.rmtree(dst_dir, ignore_errors=True)
    dst_dir.mkdir(parents=True)
    out = []
    for i, p in enumerate(files):
        with Image.open(p) as im:
            if min(im.size) < size:
                print(f"⚠️ {p.name}: {im.size} менше за {size}px — якість може постраждати")
            img = center_crop_resize(im, size)
        path = dst_dir / f"{i:03d}.png"
        img.save(path)
        out.append(path)
    return out


# Перевірка TODO 5 на синтетичному прямокутному зображенні
_test = center_crop_resize(Image.new("RGBA", (800, 600), (255, 0, 0, 128)), 512)
assert _test.size == (512, 512) and _test.mode == "RGB"

INSTANCE_DIR = WORK_DIR / "instance_images"
instance_files = prepare_instance_dir(raw_files, INSTANCE_DIR)
assert len(instance_files) >= 5, "Потрібно щонайменше 5 зображень"

preview = [Image.open(p) for p in instance_files[:15]]
show_images(preview, [p.name for p in instance_files[:15]], cols=5, size=2.6,
            suptitle=f"Навчальний датасет ({len(instance_files)} зображень)", save_as="part3_dataset.png")

### TODO 6 — промпт інстанції та гіперпараметри

Промпт інстанції складається з рідкісного ідентифікатора та **назви класу**. Клас дає моделі апріорне знання («це собака»), а ідентифікатор — конкретний екземпляр:

* об'єкт: `"a photo of sks dog"`, `"a photo of sks backpack"`, `"a photo of sks toy"`;
* стиль: `"a painting in sks style"`.

Стартові гіперпараметри нижче розраховані на 5–15 зображень і T4 (≈ 10–20 хв з урахуванням завантаження). Чекпоінти кожні 250 кроків дозволять порівняти недонавчання та перенавчання.

In [ ]:
# TODO 6: вкажіть клас вашого об'єкта (dog, cat, toy, mug, backpack, ...) або перейдіть на стиль
CLASS_NOUN = "???"
INSTANCE_TOKEN = "sks"
INSTANCE_PROMPT = f"a photo of {INSTANCE_TOKEN} {CLASS_NOUN}"
# Для стилю: INSTANCE_PROMPT = f"a painting in {INSTANCE_TOKEN} style"

assert "???" not in INSTANCE_PROMPT, "Вкажіть CLASS_NOUN"

LORA_DIR = WORK_DIR / "lora_output"
TRAIN_CFG = dict(
    resolution=512,
    train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=1e-4,        # стандарт для LoRA; для повного DreamBooth ~5e-6
    lr_scheduler="constant",
    lr_warmup_steps=0,
    max_train_steps=500,       # ~40–100 епох для 5–12 зображень
    rank=16,                   # ранг LoRA: 4–8 для стилів, 16–32 для об'єктів
    checkpointing_steps=250,
    seed=0,
    mixed_precision="fp16",
    report_to="tensorboard",
)
print("INSTANCE_PROMPT:", INSTANCE_PROMPT)

### 3.2 Запуск навчання

Використовується офіційний скрипт `train_dreambooth_lora.py` з репозиторію `diffusers` **тієї самої версії**, що встановлена в середовищі (скрипт із гілки `main` зазвичай вимагає dev-версію бібліотеки).

Перед навчанням пайплайн вивантажується з GPU: навчання запускається окремим процесом і потребує всієї VRAM.

In [ ]:
# Звільняємо GPU перед навчанням
if "pipe" in globals():
    del pipe
free_memory()

# Скрипт навчання, що відповідає встановленій версії diffusers
dv = im.version("diffusers")
ref = f"v{dv}" if "dev" not in dv else "main"
SCRIPT = WORK_DIR / "train_dreambooth_lora.py"
urllib.request.urlretrieve(
    f"https://raw.githubusercontent.com/huggingface/diffusers/{ref}/examples/dreambooth/train_dreambooth_lora.py",
    SCRIPT)
print("Скрипт:", SCRIPT, "| diffusers", dv)

# Конфігурація accelerate (1 GPU, fp16) без інтерактивних питань
from accelerate.utils import write_basic_config
write_basic_config(mixed_precision="fp16")

In [ ]:
def run_streaming(cmd, log_path):
    """Запускає процес і транслює його вивід у комірку та лог-файл."""
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    with open(log_path, "w") as log, subprocess.Popen(
            cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0) as proc:
        while True:
            chunk = proc.stdout.read(1024)
            if not chunk:
                break
            text = decoder.decode(chunk)
            sys.stdout.write(text); sys.stdout.flush()
            log.write(text)
    return proc.returncode


shutil.rmtree(LORA_DIR, ignore_errors=True)
cmd = ["accelerate", "launch", str(SCRIPT),
       f"--pretrained_model_name_or_path={MODEL_ID}",
       f"--instance_data_dir={INSTANCE_DIR}",
       f"--output_dir={LORA_DIR}",
       f"--instance_prompt={INSTANCE_PROMPT}",
       "--gradient_checkpointing",           # економія VRAM ціною ~20% швидкості
       ] + [f"--{k}={v}" for k, v in TRAIN_CFG.items()]
print(" \\\n  ".join(cmd), "\n")

t0 = time.time()
rc = run_streaming(cmd, WORK_DIR / "train_log.txt")
print(f"\nКод завершення: {rc}; час навчання: {(time.time() - t0) / 60:.1f} хв")
assert rc == 0, "Навчання завершилось з помилкою — див. вивід вище або train_log.txt"
assert (LORA_DIR / "pytorch_lora_weights.safetensors").exists()
print("Файли LoRA:", sorted(p.name for p in LORA_DIR.iterdir()))
print(f"Розмір адаптера: {(LORA_DIR / 'pytorch_lora_weights.safetensors').stat().st_size / 1e6:.1f} MB")

In [ ]:
# (Опційно) криві навчання в TensorBoard. Втрати дифузійної моделі дуже шумні:
# вони залежать від випадкового t на кожному кроці, тож не очікуйте монотонного спаду.
%load_ext tensorboard
%tensorboard --logdir /content/hw_genmodels/lora_output/logs

In [ ]:
# (Рекомендовано) збереження адаптера на Google Drive — сесія Colab може перерватися
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    shutil.copytree(LORA_DIR, "/content/drive/MyDrive/hw_genmodels_lora", dirs_exist_ok=True)
    print("Збережено. Повторне завантаження: pipe.load_lora_weights('/content/drive/MyDrive/hw_genmodels_lora')")

### 3.3 Генерація з адаптованою моделлю

### TODO 7 — промпти в нових контекстах

Складіть **щонайменше 6** промптів, що переносять ваш об'єкт у нові сцени, стилі й ситуації. Включайте ідентифікатор разом із класом (`sks dog`). Добре мати і «легкі» промпти (фото в іншому інтер'єрі), і «важкі» (інший художній стиль, незвичний контекст, аксесуари), щоб побачити межі генералізації.

In [ ]:
pipe = load_pipeline()
pipe.load_lora_weights(str(LORA_DIR), weight_name="pytorch_lora_weights.safetensors", adapter_name="final")
subject = f"{INSTANCE_TOKEN} {CLASS_NOUN}"

# TODO 7: щонайменше 6 промптів у різних контекстах; використовуйте змінну subject
LORA_PROMPTS = [
    f"a photo of {subject} ...",
    # ...
]
assert len(LORA_PROMPTS) >= 6, "Потрібно щонайменше 6 промптів"

lora_images = [generate(pipe, p, seed=i)[0] for i, p in enumerate(LORA_PROMPTS)]
show_images(lora_images, [p[:45] for p in LORA_PROMPTS], cols=3, size=4,
            suptitle="DreamBooth-LoRA: нові контексти", save_as="part3_lora_prompts.png")

### 3.4 Порівняння до / після

Однакові промпт і seed; ліворуч — базова SD 1.5 (адаптер вимкнено через `pipe.disable_lora()`), праворуч — з LoRA. Подумайте, що базова модель «розуміє» під словом `sks`.

In [ ]:
compare_prompts = [INSTANCE_PROMPT, f"a photo of {subject} sitting on a wooden table"]

before_after, titles = [], []
for p in compare_prompts:
    pipe.disable_lora()
    img_base, _ = generate(pipe, p, seed=42)
    pipe.enable_lora()
    img_lora, _ = generate(pipe, p, seed=42)
    before_after += [img_base, img_lora]
    titles += [f"SD 1.5 base\n{p}", f"SD 1.5 + LoRA\n{p}"]

show_images(before_after, titles, cols=2, size=4.5,
            suptitle="До / після адаптації (той самий seed)", save_as="part3_before_after.png")

### TODO 8 — сила адаптера (LoRA scale)

Поправку можна масштабувати під час генерації: $W = W_0 + \alpha\,BA$. При $\alpha = 0$ маємо базову модель, при $\alpha = 1$ — навчений адаптер, при $\alpha > 1$ — «підсилений».

Реалізуйте `lora_scale_sweep`: для кожного значення `scale` встановіть силу адаптера `adapter_name` і згенеруйте зображення з фіксованим seed. Силу активних адаптерів задає метод пайплайна `set_adapters` (подивіться в документації diffusers, як передати ваги адаптерів). **Наприкінці поверніть адаптеру силу 1.0**, інакше наступні комірки працюватимуть з останнім значенням.

In [ ]:
def lora_scale_sweep(pipe, adapter_name, prompt, scales, seed=42, **gen_kwargs):
    """TODO 8: повертає список зображень для кожного значення scale."""
    images = []
    raise NotImplementedError("TODO 8: реалізуйте lora_scale_sweep")
    return images


SCALES = [0.0, 0.25, 0.5, 0.75, 1.0, 1.25]
SCALE_PROMPT = f"a watercolor painting of {subject} in a flower field"
scale_images = lora_scale_sweep(pipe, "final", SCALE_PROMPT, SCALES)
show_images(scale_images, [f"scale={s}" for s in SCALES], cols=6, size=3,
            suptitle=f"Сила LoRA: «{SCALE_PROMPT}»", save_as="part3_lora_scale.png")

### 3.5 Недонавчання vs перенавчання: чекпоінти

Порівняємо адаптер після 250 і 500 кроків на двох промптах: один близький до навчальних фото, другий — у новому стилі. Перенавчання проявляється як «копіювання» фону та пози з датасету і втрата здатності виконувати промпт.

In [ ]:
adapters = {}
for ckpt in sorted(LORA_DIR.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1])):
    weights = ckpt / "pytorch_lora_weights.safetensors"
    if weights.exists():
        name = ckpt.name.replace("-", "_")
        pipe.load_lora_weights(str(ckpt), weight_name=weights.name, adapter_name=name)
        adapters[name] = ckpt.name
adapters["final"] = f"final ({TRAIN_CFG['max_train_steps']} steps)"

ckpt_prompts = [INSTANCE_PROMPT, f"a pencil sketch of {subject} reading a book"]
ckpt_images, ckpt_titles = [], []
for p in ckpt_prompts:
    for name, label in adapters.items():
        pipe.set_adapters([name], adapter_weights=[1.0])
        ckpt_images.append(generate(pipe, p, seed=3)[0])
        ckpt_titles.append(f"{label}\n{p[:40]}")
pipe.set_adapters(["final"], adapter_weights=[1.0])

show_images(ckpt_images, ckpt_titles, cols=len(adapters), size=3.5,
            suptitle="Чекпоінти навчання", save_as="part3_checkpoints.png")

### ✍️ Аналіз частини 3

Дайте розгорнуту відповідь (5–7 речень загалом, з посиланням на конкретні зображення):

1. Чи вдалося моделі засвоїти вигляд вашого об'єкта/стилю? Що саме відтворено правильно (форма, колір, текстура, характерні деталі), а що — ні?
2. Чи зберегла модель здатність виконувати нові промпти? Наведіть приклади успішних і невдалих генерацій із TODO 7. Які типи промптів виявилися найважчими?
3. Яка відмінність між базовою SD 1.5 і адаптованою моделлю на тому самому seed? Що базова модель генерує у відповідь на `sks`, і чому?
4. Як сила LoRA впливає на компроміс «схожість на об'єкт ↔ виконання промпту»? Яке значення ви обрали б на практиці?
5. Чим відрізняються чекпоінти 250 і 500 кроків? Чи бачите ознаки недонавчання або перенавчання?
6. Як guidance scale і кількість кроків впливали на якість саме у ваших генераціях з LoRA (за потреби проведіть 1–2 додаткові запуски `run_sweep`)?

✍️ **Ваші відповіді:**

1. …
2. …
3. …
4. …
5. …
6. …

## 4. Бонусні завдання (за бажанням, балів не додають)

### Бонус А — img2img і параметр `strength`

Img2img не починає з чистого шуму: вхідне зображення кодується у латент, зашумлюється до рівня `strength` і далі розшумлюється з новим промптом. Це базовий інструмент перенесення домену (наприклад, синтетика → реалістичне фото, день → ніч).

`from_pipe` створює img2img-пайплайн **на тих самих вагах** (включно з LoRA), без повторного завантаження моделі.

**TODO (бонус):** проведіть перебір `strength` для одного вхідного зображення. Замініть вхід на власне фото та промпт на свій сценарій перенесення домену.

In [ ]:
img2img = StableDiffusionImg2ImgPipeline.from_pipe(pipe)
img2img.set_progress_bar_config(disable=True)
pipe.disable_lora()    # для чистого експерименту img2img вимикаємо адаптер (спільні ваги)

init_image = Image.open(instance_files[0]).convert("RGB")   # замініть на власне зображення
I2I_PROMPT = "a painting in Van Gogh style, swirling brushstrokes, starry night"
STRENGTHS = [0.2, 0.4, 0.6, 0.8, 1.0]

i2i_images = []
for s in STRENGTHS:
    ...   # згенеруйте зображення з init_image і I2I_PROMPT при strength = s (фіксований seed)

pipe.enable_lora()
show_images([init_image] + i2i_images, ["input"] + [f"strength={s}" for s in STRENGTHS], cols=6, size=3,
            suptitle="Img2img: баланс структура ↔ промпт", save_as="bonusA_img2img.png")

### Бонус Б — кількісна оцінка: CLIP-I та CLIP-T

Візуальна оцінка суб'єктивна. У статті DreamBooth використовують дві метрики:

* **CLIP-I** — косинусна схожість CLIP-ембедінгів згенерованого зображення та навчальних зображень (*чи схоже на наш об'єкт?*);
* **CLIP-T** — схожість зображення з текстом промпту **без** ідентифікатора `sks` (*чи виконано промпт?*).

Комірка нижче обчислює обидві метрики для зображень з TODO 8 (різна сила LoRA). Проаналізуйте компроміс між ними. Пам'ятайте, що CLIP-I на 5–15 зображеннях — дуже шумна оцінка, а сам CLIP погано розрізняє схожі екземпляри одного класу (у статті DreamBooth для цього додатково використовують DINO).

In [ ]:
from transformers import CLIPModel, CLIPProcessor

clip = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").to(DEVICE).eval()
clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")


def _as_tensor(out):
    return out if isinstance(out, torch.Tensor) else out.pooler_output


@torch.no_grad()
def clip_embed_images(images):
    inputs = clip_proc(images=images, return_tensors="pt").to(DEVICE)
    return torch.nn.functional.normalize(_as_tensor(clip.get_image_features(**inputs)), dim=-1)


@torch.no_grad()
def clip_embed_text(texts):
    inputs = clip_proc(text=texts, return_tensors="pt", padding=True).to(DEVICE)
    return torch.nn.functional.normalize(_as_tensor(clip.get_text_features(**inputs)), dim=-1)


ref_emb = clip_embed_images([Image.open(p) for p in instance_files])
gen_emb = clip_embed_images(scale_images)
text_emb = clip_embed_text([SCALE_PROMPT.replace(f"{INSTANCE_TOKEN} ", "")])

clip_i = (gen_emb @ ref_emb.T).mean(dim=1).cpu().numpy()
clip_t = (gen_emb @ text_emb.T).squeeze(1).cpu().numpy()
display(pd.DataFrame({"lora_scale": SCALES, "CLIP-I (схожість на об'єкт)": clip_i.round(3),
                      "CLIP-T (відповідність промпту)": clip_t.round(3)}))

fig, ax1 = plt.subplots(figsize=(5, 3))
ax1.plot(SCALES, clip_i, "o-", color="tab:blue"); ax1.set_ylabel("CLIP-I", color="tab:blue")
ax2 = ax1.twinx(); ax2.plot(SCALES, clip_t, "s--", color="tab:orange"); ax2.set_ylabel("CLIP-T", color="tab:orange")
ax1.set_xlabel("LoRA scale"); plt.title("Компроміс схожість ↔ промпт")
plt.savefig(OUT_DIR / "bonusB_clip.png", dpi=100, bbox_inches="tight"); plt.show()

clip.to("cpu"); free_memory()   # звільняємо VRAM

### ✍️ Аналіз бонусних завдань

1. **Img2img:** як змінюється баланс між збереженням структури вхідного зображення і відповідністю промпту зі зростанням `strength`? Чим `strength=1.0` відрізняється від звичайної генерації? Як цей інструмент можна використати для sim-to-real?
2. **CLIP-метрики:** чи узгоджуються числа з вашим візуальним враженням? Які обмеження таких метрик?

✍️ **Ваші відповіді:**

1. …
2. …

## 5. Висновки

✍️ Напишіть 3–5 речень: що вдалося, що виявилося складним, яке порівняння підходів ви можете зробити (семплери, сила LoRA, чекпоінти) і **чим тонке налаштування LoRA відрізняється від навчання генеративної моделі з нуля** (дані, обчислення, що саме навчається, ризики).

✍️ **Ваші висновки:**

…

In [ ]:
# Архів з усіма фігурами, таблицею часу та логом навчання — покладіть його в архів здачі разом із ноутбуком
shutil.copy(WORK_DIR / "train_log.txt", OUT_DIR / "train_log.txt")
archive = shutil.make_archive(str(WORK_DIR / "hw_outputs"), "zip", OUT_DIR)
print("Архів:", archive)
print(sorted(p.name for p in OUT_DIR.iterdir()))
# щоб скачати архів, розкоментуйте рядок нижче (або знайдіть файл у панелі Files)
# from google.colab import files; files.download(archive)

## 6. Ідеї для самостійних експериментів

Ці пункти не обов'язкові, але добре показують, що відбувається «під капотом», і безпосередньо пов'язані з темами заняття 8.

**Навчання адаптера**
* **Prior preservation.** Додайте `--with_prior_preservation --class_data_dir=... --class_prompt="a photo of dog" --num_class_images=100`. Скрипт згенерує зображення класу базовою моделлю й регуляризуватиме ними навчання. Порівняйте, чи рідше «всі собаки стають вашою собакою».
* **Сітка гіперпараметрів.** `rank ∈ {4, 16, 64}`, `learning_rate ∈ {5e-5, 1e-4, 3e-4}`, `max_train_steps ∈ {250, 500, 1000}`. Як змінюються розмір адаптера, схожість і перенавчання?
* **Навчання текстового енкодера.** Прапорець `--train_text_encoder` часто покращує схожість облич і складних об'єктів.
* **Textual Inversion** (`examples/textual_inversion`) — навчається лише один ембедінг токена. Порівняйте з LoRA за якістю, розміром і часом.
* **Два адаптери одночасно.** Натренуйте LoRA об'єкта й LoRA стилю і скомбінуйте: `pipe.set_adapters(["obj", "style"], adapter_weights=[1.0, 0.7])`.

**Керування генерацією**
* **ControlNet** (`lllyasviel/sd-controlnet-canny`, `-depth`, `-openpose`) разом із вашою LoRA: об'єкт у заданій позі або композиції.
* **IP-Adapter** (`pipe.load_ip_adapter("h94/IP-Adapter", subfolder="models", weight_name="ip-adapter_sd15.bin")`) — адаптація за одним референсним зображенням **без навчання**. Порівняйте з DreamBooth-LoRA.
* **Inpainting** (`stable-diffusion-v1-5/stable-diffusion-inpainting`) — вставлення вашого об'єкта в реальне фото за маскою.
* **Швидкі семплери.** LCM-LoRA (`latent-consistency/lcm-lora-sdv1-5`) — генерація за 4 кроки; порівняйте якість і час з експериментом Г.

**Генеративні моделі як джерело даних**
* Згенеруйте 100–200 зображень рідкісного класу для будь-якого детектора або класифікатора й порівняйте метрики на **реальному** тестовому наборі в трьох режимах: тільки реальні дані, реальні + синтетика, тільки синтетика. Не використовуйте синтетичні зображення у валідації та тесті — інакше метрика буде завищеною.
* Спробуйте фільтрувати синтетику за CLIP-T і подивіться, як це впливає на результат.
* Оцініть distribution shift між реальними та синтетичними даними (наприклад, FID або відстань між CLIP-ембедінгами).

**Сучасніші моделі**
* SDXL або FLUX.1-schnell (на T4 — з `enable_model_cpu_offload()` / `enable_sequential_cpu_offload()`): порівняйте відтворення тексту, анатомію та виконання складних промптів із SD 1.5. Для SDXL існує аналогічний скрипт `train_dreambooth_lora_sdxl.py`.

---
Після завершення роботи: *Runtime → Disconnect and delete runtime*, щоб звільнити GPU.